BB84 Quantum Key Distribution (QKD) Protocol Simulation
=========================================================
Simulates the full BB84 protocol between Alice and Bob,
with an optional eavesdropper (Eve) in the middle.
 
Protocol steps:
  1. Alice prepares qubits in randomly chosen bases (computational or hadamard) with random bits (0/1).
  2. Optionally, Eve intercepts and measures in random bases.
  3. Bob measures in random bases.
  4. Alice and Bob publicly announce their bases (not bits) via classical channel.
  5. They keep only bits where bases matched ie sifted key.
  6. They compare a sample of the sifted key to estimate Quantum Bit Error Rate (QBER) and detect eavesdropping based on threshold violation.

In [116]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
 
# simulation configurations
NUM_BITS = 200       # no of qubits Alice sends
SAMPLE_FRACTION = 0.2       # fraction of sifted key used for QBER check
EVE_PRESENT = True      # eavesdropping on/off
SEED = 69        

rng=np.random.default_rng(SEED) # random number generator for reproducibility
simulator = AerSimulator() # using the Aer simulator for running circuits since we dont have access to a real quantum hardware

In [117]:
# build one BB84 qubit circuit : prepare and measure a single qubit

def prepare_qubit(bit: int, basis: str) -> QuantumCircuit:
    """
    Alice prepares a single qubit.
      basis '+' (computational/Z): {|0⟩, |1⟩}
      basis 'x' (diagonal/hadamard):   {|+⟩, |−⟩}
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)  # flip to |1⟩
    if basis == 'x':
        qc.h(0)   # rotate to diagonal (hadamard) basis
    return qc

def measure_qubit(qc: QuantumCircuit, basis: str) -> QuantumCircuit:
    """
    Append a measurement in the chosen basis onto an existing circuit.
      basis '+': measure in Z basis (standard)
      basis 'x': rotate back with H then measure
    Returns a NEW circuit (copy + measurement) to avoid mutating the original.
    """
    qc2 = qc.copy()
    if basis == 'x':
        qc2.h(0)
    qc2.measure(0, 0)
    return qc2


def run_circuit(qc: QuantumCircuit) -> int:
    """Execute a single-qubit circuit and return the measured bit (0 or 1)."""
    job    = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0])  # '0' or '1' → int

In [118]:
# sanity check: prepare |+⟩ and measure in Z basis should yield 0 or 1 with ~50% probability

N=1000
n=0
for _ in range(N):
    xx=prepare_qubit(1, 'x')
    # display(xx.draw('mpl', initial_state=True))
    if run_circuit(measure_qubit(xx, '+')):
        n+=1
print(f"Prepared |+⟩ and measured |1> in Z basis {n/N:.2%} times (should be ~50%)")

Prepared |+⟩ and measured |1> in Z basis 50.40% times (should be ~50%)


In [119]:
# sanity check: prepare |-⟩ and measure in Z basis should yield 0 or 1 with ~50% probability

N=1000
n=0
for _ in range(N):
    xx=prepare_qubit(0, 'x')
    # display(xx.draw('mpl', initial_state=True))
    if run_circuit(measure_qubit(xx, '+')):
        n+=1
print(f"Prepared |−⟩ and measured |1> in Z basis {n/N:0.2%} of time (should be ~50%)")

# sanity check: prepare |+⟩ and measure in H basis should yield |+> with 100% probability

n=0
for _ in range(N):
    xx=prepare_qubit(1, '+')
    # display(xx.draw('mpl', initial_state=True))
    if run_circuit(measure_qubit(xx, '+')):
        n+=1
print(f"Prepared |+> and measured |+> in H basis {n/N:0.2%} of time (should be 100%)")

# sanity check: prepare |0⟩ and measure in Z basis should yield 0 with 100% probability

n=0
for _ in range(N):
    xx=prepare_qubit(0, '+')
    # display(xx.draw('mpl', initial_state=True))
    if not run_circuit(measure_qubit(xx, '+')):
        n+=1
print(f"Prepared |0> and measured |0> in Z basis {n/N:0.2%} of time (should be 100%)")



Prepared |−⟩ and measured |1> in Z basis 48.30% of time (should be ~50%)
Prepared |+> and measured |+> in H basis 100.00% of time (should be 100%)
Prepared |0> and measured |0> in Z basis 100.00% of time (should be 100%)


Let us now start the actual BB84 protocol:

In [120]:
# step 1 – Alice generates random bits & bases

alice_bits = [rng.integers(0, 2) for _ in range(NUM_BITS)]
alice_bases = [rng.choice(['+', 'x']) for _ in range(NUM_BITS)]
 
# build Alice's qubits (circuits without measurement)
alice_circuits = [prepare_qubit(bit, basis) for bit, basis in zip(alice_bits, alice_bases)]

In [121]:
# step 2 – eavesdropper Eve intercepts (optional)

eve_bases = [rng.choice(['+', 'x']) for _ in range(NUM_BITS)]
eve_bits = [] # eve's measurement results
 
circuits_to_bob = []  # what Bob will actually receives
 
if EVE_PRESENT:
    for a_ckt, e_base in zip(alice_circuits, eve_bases):
        # Eve measures in her random basis
        eve_qc  = measure_qubit(a_ckt, e_base)
        eve_bit = run_circuit(eve_qc)
        eve_bits.append(eve_bit)
 
        # Eve re-prepares the qubit based on her measurement result (introducing errors as E doenst know Alice's basis)
        new_qc = prepare_qubit(eve_bit, e_base)
        circuits_to_bob.append(new_qc)
else:
    circuits_to_bob = alice_circuits  # qubit travels undisturbed

In [122]:
# step 3 - Bob measures in random bases

bob_bases = [rng.choice(['+', 'x']) for _ in range(NUM_BITS)]
bob_bits = [] # bob's measurement results
 
for ckt, base in zip(circuits_to_bob, bob_bases):
    bob_qc  = measure_qubit(ckt, base)
    bob_bit = run_circuit(bob_qc)
    bob_bits.append(bob_bit) 

In [123]:
# step 4 – basis reconciliation (via public channel) 
# A and B compare bases; keep matching indices

sifted_indices = [i for i in range(NUM_BITS) if alice_bases[i] == bob_bases[i]]
 
alice_sifted = [alice_bits[i] for i in sifted_indices]
bob_sifted   = [bob_bits[i]   for i in sifted_indices]
 
sifted_len = len(sifted_indices)

In [124]:
# step 5 – QBER estimation (sacrifice a sample)

sample_size    = max(1, int(sifted_len * SAMPLE_FRACTION))
sample_indices = rng.choice(a=range(sifted_len), size=sample_size, replace=False)
 
errors = sum(
    1 for i in sample_indices
    if alice_sifted[i] != bob_sifted[i]
)
qber = errors / sample_size
 
# final key uses the NON-sampled bits

key_indices = [i for i in range(sifted_len) if i not in set(sample_indices)]
alice_key   = [alice_sifted[i] for i in key_indices]
bob_key     = [bob_sifted[i]   for i in key_indices] 

In [125]:
# step 6 – security decision
# Theoretical QBER thresholds:
#   0%      → no eavesdropping
#   ~25%    → Eve measuring all bits (expected QBER ≈ 25%)
#   > 11%   → BB84 security threshold (key is compromised)

SECURITY_THRESHOLD = 0.11   
key_secure = qber <= SECURITY_THRESHOLD
 
key_agreement = sum(a == b for a, b in zip(alice_key, bob_key)) / len(alice_key) if alice_key else 0.0

In [134]:
# summary

print("=" * 50)
print("BB84 QKD Simulation — Results")
print("=" * 50)
print(f"Qubits sent by Alice  : {NUM_BITS}")
print(f"Eve present           : {EVE_PRESENT}\n")

print(f"Sifted key length     : {sifted_len}  "
      f"({100*sifted_len/NUM_BITS:.0f}% of qubits — expected ~50%)")
print(f"Sample for QBER check : {sample_size} bits "
      f"({100*SAMPLE_FRACTION:.0f}% of sifted)")
print(f"Errors in sample      : {errors}")
print(f"QBER                  : {qber*100:.1f}%  "
      f"(threshold: {SECURITY_THRESHOLD*100:.0f}%)\n")

print(f"Final key length      : {len(alice_key)} bits")
print(f"Key agreement (Alice=Bob) : {key_agreement*100:.1f}%\n")
if key_secure:
    print("SECURE — QBER is below threshold. Key accepted.")
else:
    print("COMPROMISED — QBER exceeds threshold. Key discarded.")
    print("Eve likely intercepted the channel.")
print("=" * 50)
 
# how often Eve guessed the right basis
if EVE_PRESENT:
    eve_basis_match = sum(
        1 for i in range(NUM_BITS) if eve_bases[i] == alice_bases[i]
    )
    print(f"Eve guessed Alice's basis correctly: "
          f"{eve_basis_match}/{NUM_BITS} times "
          f"({eve_basis_match/NUM_BITS:.0%} — expected ~50%)")

BB84 QKD Simulation — Results
Qubits sent by Alice  : 200
Eve present           : True

Sifted key length     : 99  (50% of qubits — expected ~50%)
Sample for QBER check : 19 bits (20% of sifted)
Errors in sample      : 5
QBER                  : 26.3%  (threshold: 11%)

Final key length      : 80 bits
Key agreement (Alice=Bob) : 80.0%

COMPROMISED — QBER exceeds threshold. Key discarded.
Eve likely intercepted the channel.
Eve guessed Alice's basis correctly: 96/200 times (48% — expected ~50%)


## Refactoring the code

We will refactor the above code into a much more reuable class structure.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from dataclasses import dataclass

# constants
BASES = ['+', 'x']  # '+' for Z basis, 'x' for H basis
SECURITY_THRESHOLD = 0.11   # QBER threshold for a secure key

# simulation configurations class
@dataclass
class Config_BB84:
    num_bits : int = 200       # no of qubits Alice sends
    sample_fraction : float = 0.2       # fraction of sifted key used for QBER check
    eve_present : bool = True      # eavesdropping on/off
    seed : int = 69       

# results class of relevant data from the simulation
@dataclass
class Result_BB84:
    alice_bits: list[int]
    alice_bases: list[str]
    bob_bits: list[int]
    bob_bases: list[str]
    alice_key: list[int]
    bob_key: list[int]
    eve_bases: list[str]
    eve_bits: list[int]
    sifted_indices: list[int]
    sample_indices: list[int]
    errors: int
    qber: float
    key_secure: bool
    key_agreement: float

# helper functions to build circuits, run them, and generate random bits/bases
def prepare_qubit(bit: int, basis: str) -> QuantumCircuit:
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == "x":
        qc.h(0)
    return qc    

def measure_qubit(qc: QuantumCircuit, basis: str) -> QuantumCircuit:
    measured = qc.copy()
    if basis == "x":
        measured.h(0)
    measured.measure(0, 0)
    return measured

def run_circuit(simulator: AerSimulator, qc: QuantumCircuit) -> int:
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(next(iter(counts)))

def random_bits(rng: np.random.Generator, size: int) -> list[int]:
    return rng.integers(0, 2, size=size).tolist()

def random_bases(rng: np.random.Generator, size: int) -> list[str]:
    return rng.choice(BASES, size=size).tolist()

In [ ]:
# main function to run the BB84 simulation with a given configuration and return results

def run_bb84(config: Config_BB84) -> Result_BB84:
    rng = np.random.default_rng(config.seed)
    simulator = AerSimulator()

    alice_bits = random_bits(rng, config.num_bits)
    alice_bases = random_bases(rng, config.num_bits)
    alice_circuits = [
        prepare_qubit(bit, basis) for bit, basis in zip(alice_bits, alice_bases)
    ]

    eve_bases = random_bases(rng, config.num_bits)
    eve_bits: list[int] = []
    circuits_to_bob: list[QuantumCircuit] = []

    if config.eve_present:
        for circuit, eve_basis in zip(alice_circuits, eve_bases):
            eve_measurement = measure_qubit(circuit, eve_basis)
            eve_bit = run_circuit(simulator, eve_measurement)
            eve_bits.append(eve_bit)
            circuits_to_bob.append(prepare_qubit(eve_bit, eve_basis))
    else:
        circuits_to_bob = alice_circuits

    bob_bases = random_bases(rng, config.num_bits)
    bob_bits = [
        run_circuit(simulator, measure_qubit(circuit, basis))
        for circuit, basis in zip(circuits_to_bob, bob_bases)
    ]

    sifted_indices = [
        index
        for index, (alice_basis, bob_basis) in enumerate(zip(alice_bases, bob_bases))
        if alice_basis == bob_basis
    ]
    alice_sifted = [alice_bits[index] for index in sifted_indices]
    bob_sifted = [bob_bits[index] for index in sifted_indices]

    if not alice_sifted:
        sample_indices: list[int] = []
        errors = 0
        qber = 0.0
        alice_key = []
        bob_key = []
    else:
        sample_size = max(1, int(len(alice_sifted) * config.sample_fraction))
        sample_size = min(sample_size, len(alice_sifted))
        sample_indices = sorted(
            rng.choice(len(alice_sifted), size=sample_size, replace=False).tolist()
        )
        sample_index_set = set(sample_indices)
        errors = sum(
            1
            for index in sample_indices
            if alice_sifted[index] != bob_sifted[index]
        )
        qber = errors / sample_size
        alice_key = [
            bit for index, bit in enumerate(alice_sifted) if index not in sample_index_set
        ]
        bob_key = [
            bit for index, bit in enumerate(bob_sifted) if index not in sample_index_set
        ]

    key_secure = qber <= SECURITY_THRESHOLD
    key_agreement = (
        sum(a == b for a, b in zip(alice_key, bob_key)) / len(alice_key)
        if alice_key
        else 0.0
    )

    return Result_BB84(
        alice_bits=alice_bits,
        alice_bases=alice_bases,
        bob_bits=bob_bits,
        bob_bases=bob_bases,
        alice_key=alice_key,
        bob_key=bob_key,
        eve_bases=eve_bases,
        eve_bits=eve_bits,
        sifted_indices=sifted_indices,
        sample_indices=sample_indices,
        errors=errors,
        qber=qber,
        key_secure=key_secure,
        key_agreement=key_agreement,
    )

In [ ]:
# helper function to print results

def print_results(config, result):
    print("=" * 50)
    print("BB84 QKD Simulation — Results")
    print("=" * 50)
    print(f"Qubits sent by Alice  : {config.num_bits}")
    print(f"Eve present           : {config.eve_present}\n")

    print(f"Sifted key length     : {len(result.sifted_indices)}  "
        f"({100*len(result.sifted_indices)/config.num_bits:.0f}% of qubits — expected ~50%)")
    print(f"Sample for QBER check : {len(result.sample_indices)} bits "
        f"({100*config.sample_fraction:.0f}% of sifted)")
    print(f"Errors in sample      : {result.errors}")
    print(f"QBER                  : {result.qber*100:.1f}%  "
        f"(threshold: {SECURITY_THRESHOLD*100:.0f}%)\n")

    print(f"Final key length      : {len(result.alice_key)} bits")
    print(f"Key agreement (Alice=Bob) : {result.key_agreement*100:.1f}%\n")
    if result.key_secure:
        print("SECURE — QBER is below threshold. Key accepted.")
    else:
        print("COMPROMISED — QBER exceeds threshold. Key discarded.")
        print("Eve has likely intercepted the channel.")
    print("=" * 50)
    
    # how often Eve guessed the right basis
    if config.eve_present:
        eve_basis_match = sum(
            1 for i in range(config.num_bits) if result.eve_bases[i] == result.alice_bases[i]
        )
        print(f"Eve guessed Alice's basis correctly: "
            f"{eve_basis_match}/{config.num_bits} times "
            f"({eve_basis_match/config.num_bits:.0%} — expected ~50%)")

Now that we are done refactoring, we can run the simulations. First we will run it with 300 bits, with eavesdropping.

In [ ]:
# run the simulation with 300 bits, Eve present, and the default random seed
config = Config_BB84(num_bits=300)
result = run_bb84(config)
print_results(config, result)


BB84 QKD Simulation — Results
Qubits sent by Alice  : 300
Eve present           : True

Sifted key length     : 142  (47% of qubits — expected ~50%)
Sample for QBER check : 28 bits (20% of sifted)
Errors in sample      : 6
QBER                  : 21.4%  (threshold: 11%)

Final key length      : 114 bits
Key agreement (Alice=Bob) : 81.6%

COMPROMISED — QBER exceeds threshold. Key discarded.
Eve has likely intercepted the channel.
Eve guessed Alice's basis correctly: 152/300 times (51% — expected ~50%)


We will run another simluation but this time with 500 bits and no Eve.

In [206]:
# run the simulation with the no eve
config = Config_BB84(num_bits=500,eve_present=False, seed=13)
result = run_bb84(config)
print_results(config, result)

BB84 QKD Simulation — Results
Qubits sent by Alice  : 500
Eve present           : False

Sifted key length     : 253  (51% of qubits — expected ~50%)
Sample for QBER check : 50 bits (20% of sifted)
Errors in sample      : 0
QBER                  : 0.0%  (threshold: 11%)

Final key length      : 203 bits
Key agreement (Alice=Bob) : 100.0%

SECURE — QBER is below threshold. Key accepted.
